In [48]:
# Project Name: Next-Out
# Description: Summarize data from many files into a single Dataframe or Excel file. Export out a new "Next-Out" file (*.no) to reduce post-processing time. 
# Copyright (c) 2024 Justin Edenbaum, Never Gray
#
# This file is licensed under the MIT License.
# You may obtain a copy of the license at https://opensource.org/licenses/MIT

from pathlib import Path
import NO_parser
import pandas as pd
import gzip
import pickle

# This function creates a Next-Out file, which is a compressed pickle
# You need to use the same version of Python that created the NO file. For Next-Out Version 1.4, you need Python 3.13.


In [49]:
def create_no_file(output_file_path):
    # Parse the file
    data, output_meta_data = NO_parser.parse_file(output_file_path, gui="", conversion_setting="SI")
    
    # Define the output file path with the .no suffix
    no_file_path = output_file_path.with_suffix('.no')
    
    # Save the variable 'data' to the file with compression
    with gzip.open(no_file_path, 'wb') as file:
        no_file = {'output_file_path': output_file_path, 'data': data, 'output_meta_data': output_meta_data}
        pickle.dump(no_file, file)
    
    print(f"{no_file_path} created.")

In [50]:
# Create NO Files for all SES output files in a directory
def create_no_files(directory_path):
    # Iterate over all files with a suffix of '.out' in the directory
    for output_file_path in directory_path.glob('*.out'):
        create_no_file(output_file_path)

In [51]:
def read_no_file(no_file_path):
    with gzip.open(no_file_path, 'rb') as file:
        no_file = pickle.load(file)
    data = no_file['data']
    output_meta_data = no_file['output_meta_data']
    return data, output_meta_data

In [52]:
def get_fire_location_and_airflow(data, output_meta_data, last_time_value=None):
    # Load the variable 'data' and 'output_meta_data' from the pickled file
    if last_time_value is None:
        last_time_value = data['SSA'].index.get_level_values('Time').max()
    fire_segment = int(output_meta_data['form4_df'].index.get_level_values('Segment')[0])
    airflow_value = data['SSA'].loc[(last_time_value, fire_segment), 'Airflow']

    return fire_segment, airflow_value

In [53]:
def get_airflow(data, segment, last_time_value=None):
    # Load the variable 'data' and 'output_meta_data' from the pickled file
    if last_time_value is None:
        last_time_value = data['SSA'].index.get_level_values('Time').max()
    airflow_value = data['SSA'].loc[(last_time_value, segment), 'Airflow']

    return airflow_value

In [54]:
def get_real_airflow(data, segment, last_time_value=None):
    # Load the variable 'data' and 'output_meta_data' from the pickled file
    if last_time_value is None:
        last_time_value = data['SST'].index.get_level_values('Time').max()
    sub_segment = 1
    Actual_Airflow_NO = data['SST'].loc[(last_time_value, segment, sub_segment), 'Actual_Airflow_NO']
    Air_Temp = data['SST'].loc[(last_time_value, segment, sub_segment), 'Air_Temp']

    return Actual_Airflow_NO, Air_Temp

In [55]:
#Summarize all airflows at the fire site. Only works for files with a fire. 
def summarize_fire_airflow(no_directory_path):
    summary_data = []
    # Search for *.no in the directory
    for file_path in no_directory_path.glob('*.no'):
        data, output_meta_data = read_no_file(file_path)
        try:
            # Get the fire segment and airflow value
            fire_segment, airflow_value = get_fire_location_and_airflow(data, output_meta_data)
            
            # Append the data to the summary list
            summary_data.append({
                'File Name': file_path.stem,
                'Fire Segment': fire_segment,
                'Airflow': abs(airflow_value)
            })
        except:
            print(f"Error processing {file_path}")

    # Create a DataFrame from the summary data
    summary_df = pd.DataFrame(summary_data)

    # Display the summary DataFrame
    print(summary_df)
    # Save the summary DataFrame as an Excel file in the same directory
    summary_df.to_excel(no_directory_path / 'fire_summary.xlsx', index=False)


In [56]:
#Summarizes airflow for multiple segment lookups at different segments for all NO Files in a directory
def summarize_airflow_4_segments(no_directory_path, segment_lookup, Actual_Airflow=False):
    summary_data = []

    # Iterate over all files with a suffix of '.no' in the directory
    for no_file_path in no_directory_path.glob('*.no'):
        # Get the fire segment and airflow value
        print(no_file_path)
        data, output_meta_data = read_no_file(no_file_path)
        for key in segment_lookup:
            airflow_value = get_airflow(data, key)
            if Actual_Airflow:
                # Get the actual airflow and air temperature if Actual_Airflow is True  
                Actual_Airflow_NO, Air_Temp = get_real_airflow(data, key)
            if airflow_value is not None and not Actual_Airflow:
                # Append the data to the summary list
                summary_data.append({
                    'File Name': no_file_path.stem,
                    'Segment': segment_lookup[key],
                    'SES Airflow': (airflow_value),
                    'Actual Airflow NV': Actual_Airflow_NO,
                    'Air Temp': Air_Temp
                })
            elif Actual_Airflow_NO is not None:
                summary_data.append({
                    'File Name': no_file_path.stem,
                    'Segment': key,
                    'SES_Airflow': (airflow_value),
                    'Actual_Airflow_NO': Actual_Airflow_NO,
                    'Air_Temp': Air_Temp
                })
    summary_df = pd.DataFrame(summary_data)
    # Create a DataFrame from the summary data
    #summary_df.set_index(['File Name', 'Segment'], inplace=True)
    # Save to Excel
    summary_df.to_excel(no_directory_path / 'summary.xlsx')
    return summary_df


In [58]:
from pathlib import Path
no_directory_path = Path('C:\\Users\\6019997\\OneDrive - Gruppo Ferrovie Dello Stato\\TVS-FLS Task\\SES-PTUS\\Calculations\\SES-277 Stairway Pressurization')
segment_lookup = [800,802]
summary_df = summarize_airflow_4_segments(no_directory_path, segment_lookup, Actual_Airflow=True)

C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-001.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-002-F.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-002-WG-F.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-002-WM-F.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-002.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-003-F.no
C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Calculations\SES-277 Stairway Pressurization\PT09-S1GM-003.no
C:\Use

In [59]:

# Pivot the DataFrame so each segment's values become columns
pivot_df = summary_df.pivot(index='File Name', columns='Segment', values=['SES_Airflow'])

# Flatten the MultiIndex columns for easier use
pivot_df.columns = [f"{col[0]}_{col[1]}" for col in pivot_df.columns]

# Save to Excel
pivot_df.to_excel(no_directory_path / 'pivot_segment_summary.xlsx')

In [ ]:
from pathlib import Path 
segement_lookup=[904,909,913,924,939]

#Files from C:\Users\6019997\OneDrive - Gruppo Ferrovie Dello Stato\TVS-FLS Task\SES-PTUS\Simulations\PT07-B001\_SIMULATION FOR CFD BOUNDARY are copied ot the directory below
directory_path = Path('C:\\Simulations\\_SIMULATION FOR CFD BOUNDARY')
#create_no_files(directory_path)
#summarize_fire_airflow(directory_path)
summarize_airflow_4_segments(directory_path, segment_lookup, Actual_Airflow=True)